In [0]:
pip install jinja2

In [0]:
from jinja2 import Template

In [0]:
parameters =[{
                "table":"project_catalog.silver.factstream",
                "alias" :"fact",
                "cols" : "fact.stream_id,fact.listen_duration"},
             {
                 "table":"project_catalog.silver.dimuser",
                "alias" :"user",
                "cols" : "user.user_id,user.user_name",
                "condition" : "fact.user_id = user.user_id"
             },
             {
                 "table":"project_catalog.silver.dimtrack",
                "alias" :"track",
                "cols" : "track.track_id,track.track_name",
                "condition" : "fact.track_id = track.track_id"
             }          
             
             ]

In [0]:
query_text = """
                select 
                        {% for param in parameters %}
                            {{ param.cols }}
                                {% if not loop.last %}
                                ,
                                {% endif %}
                        {% endfor %}
                from    
                        {% for param in parameters %}
                            {% if loop.first %}
                                {{ param['table'] }} as  {{ param['alias'] }}
                            {% endif %}
                        {% endfor %}                       
                        {% for param in parameters %}
                            {% if not loop.first %}
                                LEFT JOIN
                                    {{ param['table'] }} as  {{ param['alias'] }}
                                ON
                                    {{param['condition'] }}
                            {% endif %}
                        {% endfor %}
"""

In [0]:
jinja_sql_str = Template(query_text)
query = jinja_sql_str.render(parameters = parameters)
print(query)

In [0]:
display(spark.sql(query))